## DESIRT light curve classification
This notebook applies a boosting classifier to FPCA-fitted light curves to identify Type Ia supernova (SNIa) candidates. An object is written to SNIa_DESIRT_names.csv if its predicted SNIa probability exceeds a specified threshold in at least m out of n trials.

###  Imports

In [27]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score
)
from catboost import CatBoostClassifier
import random


### Configuration

In [28]:
n=5
m=3
thres=0.5
random_seeds = random.sample(range(0, 100), n)
output_file="SNIa_DESIRT_names.csv"

### Load Data

In [29]:
# Load dataset
df_train = pd.read_csv('plasticc_train_data.csv')
df_test = pd.read_csv('desirt_final_data.csv')

# Show unique transient types
print("Printing Transient Types:", df_train['type'].unique())

# Group all non-SN Ia as 'Non-SNIa'
df_train.loc[df_train['type'] != 'SN Ia', 'type'] = 'Non-SNIa'

df_train = df_train.rename(columns={
    'g_alpha': 'g_a1', 'g_beta': 'g_a2',
    'r_alpha': 'r_a1', 'r_beta': 'r_a2',
    'z_alpha': 'z_a1', 'z_beta': 'z_a2'
})

Printing Transient Types: ['SN Ia' 'SN II' 'SLSN' 'SN Ibc' 'SN Ia-91bg' 'ILOT' 'CaRT' 'SN Iax' 'TDE'
 'Point source μ-lensing' 'PISN' 'Binary system μ-lensing']


###  Define Features and Labels

In [30]:
feature_cols = [
    'g_pk_mag','r_pk_mag', 'z_pk_mag', 
    'g_a1', 'g_a2', 'r_a1', 'r_a2', 'z_a1', 'z_a2'
]
label_col = 'type'

###  Define Training, Validation and Test Set


In [31]:
x_train_val = df_train[feature_cols].to_numpy()
y_train_val = df_train[label_col].to_numpy()

x_test = df_test[feature_cols].to_numpy()
x_name=df_test['name'].to_numpy()


x_train, x_val, y_train, y_val = train_test_split(
    x_train_val, y_train_val, test_size=0.1, random_state=90
)

print('Training data size:', np.shape(x_train)[0])
print('Test data size:', np.shape(x_test)[0])
print('No. of features:', np.shape(x_train)[1])


Training data size: 8033
Test data size: 113
No. of features: 9


###  Train CatBoost 

In [32]:
vote_count = {}

for j in range(len(random_seeds)):
    print(f"{j+1}/{n} runs")
    clf = CatBoostClassifier(random_seed=random_seeds[j], verbose=0)
    clf.fit(x_train, y_train, eval_set=(x_val, y_val))
    pr = clf.predict(x_test, prediction_type='Probability')
    
    for i in range(len(x_test)):
        if pr[i][1] > thres:
            vote_count[i] = vote_count.get(i, 0) + 1

SNIa_names = [x_name[i] for i, count in vote_count.items() if count >= m]
pd.DataFrame({'name': SNIa_names}).to_csv(output_file, index=False)
print(len(SNIa_names))

1/5 runs
2/5 runs
3/5 runs
4/5 runs
5/5 runs
57
